# Análisis Maestro - Matriz de Control 24 de Abril

Este cuaderno documenta el proceso completo de integración de datos para la generación del reporte de control de Regionales de ICBF.

## El Desafío de los Identificadores (NIT Bridge)
El reto principal de este cruce radica en que **Regionales** utiliza identificadores de contrato de la vigencia 2026, mientras que el **Metaverso Auditado** utiliza identificadores del 2025. Al no existir un identificador de contrato común, se implementó una estrategia de **Puente de Datos (NIT Bridge)**:
1. Se utiliza la base **Cuéntame** para mapear el ID de Contrato 2026 hacia el **NIT del Contratista**.
2. Con el NIT, se consolida la información del **Metaverso** y la **Zonificación**.
3. Se aplica **Fuzzy Matching** sobre los nombres de los servicios para asignar correctamente los cupos a los 7 slots horizontales de la base original.

### 1. Configuración de Entorno y Funciones de Limpieza

In [1]:
import pandas as pd
import re
import os
import xlsxwriter
from thefuzz import process, fuzz

# Configuración de Rutas Relativas
PATHS = {
    'regionales': r"../data/insumos matriz 24 abril/Regionales Contratacion Vigente 2026_CONSOLIDADO.xlsx",
    'compromisos': r"../data/insumos matriz 24 abril/LISTADO DE COMPROMISOS NACION 20042026.xlsx",
    'cuentame': r"../data/insumos matriz 24 abril/ICBFCUEContUnicoxRegxVigxDir.xlsx",
    'metaverso': r"../data/insumos metaverso/RECONSTRUCCION_METAVERSO_2026_NACIONAL_AUDITADO.xlsx",
    'plantilla': r"../data/Plantilla_Matriz_Adición_Nivelacion_ V3_24032026_paradividir_correcion.xlsx"
}

def clean_id(val):
    """Limpia IDs para dejar solo dígitos, vital para cruce estricto de NITs y Contratos."""
    if pd.isna(val): return ""
    return re.sub(r'\D', '', str(val))
def normalize_regional(val):
    """Estandariza nombres de regionales para cruces consistentes."""
    if pd.isna(val): return ""
    val = str(val).upper()
    val = re.sub(r'ICBF DIRECCI.N REGIONAL ', '', val)
    val = re.sub(r'ICBF REGIONAL ', '', val)
    val = re.sub(r'REGIONAL ', '', val)
    return val.strip()


### 2. Extracción y Construcción del Puente (NIT Bridge)
Procesamos Compromisos (Auditoría Financiera) y Cuéntame (Puente de NITs).

In [2]:
print("Cargando fuentes de datos primarias...")

# 1. Regionales (Base)
df_reg = pd.read_excel(PATHS['regionales'], sheet_name='Tabla1')
df_reg['id_clean'] = df_reg['Numero Documento Soporte'].apply(clean_id)
df_reg['reg_clean'] = df_reg['Entidad Descripcion'].apply(normalize_regional)

# 2. Compromisos
df_comp_full = pd.read_excel(PATHS['compromisos'])
col_t_name = df_comp_full.columns[19]
df_comp = df_comp_full[df_comp_full[col_t_name] == 161].copy()
df_comp['id_clean'] = df_comp['Numero Documento Soporte'].apply(clean_id)
fin_cols = ['Valor Actual', 'Saldo por Obligar', 'Valor_Inicial']
df_comp_agg = df_comp.groupby('id_clean')[fin_cols].sum().reset_index()

# 3. Cuentame
df_cue = pd.read_excel(PATHS['cuentame'])
df_cue['id_clean'] = df_cue['Numero contrato'].apply(clean_id)
df_cue['reg_clean'] = df_cue['Regional del contrato'].apply(normalize_regional)
col_cue_nit = [c for c in df_cue.columns if 'identificac' in str(c).lower()][0]
df_cue['nit_clean'] = df_cue[col_cue_nit].apply(clean_id)

df_cue_nit_map = df_cue[['id_clean', 'nit_clean', 'reg_clean']].dropna(subset=['nit_clean']).drop_duplicates()
df_cue_agg = df_cue.groupby('id_clean')['Numero Cupos'].sum().reset_index().rename(columns={'Numero Cupos': 'Atenciones Cuentame'})


Cargando fuentes de datos primarias...


### 3. Preparación de Metaverso y Zonificación por NIT
Adaptamos las bases de auditoría para que su llave primaria sea el NIT del contratista, lo que nos permitirá agrupar los servicios ofertados.

In [3]:
# 4. Metaverso (Auditado)
df_met = pd.read_excel(PATHS['metaverso'])
df_met['nit_clean'] = df_met['unive'].apply(clean_id)
df_met['reg_clean'] = df_met['Regional UDS'].apply(normalize_regional)
df_met['Codigo UDS'] = df_met['Codigo Unidad Servicio UDS'].astype(str).str.split('.').str[0].str.strip()
df_met['Nombre UDS'] = df_met['Unidad Servicio UDS']

# 5. Zonificacion (Plantilla)
xl_plan = pd.ExcelFile(PATHS['plantilla'])
sheet_zon = [s for s in xl_plan.sheet_names if 'ZONIFICACI' in s][0]
df_zon_raw = xl_plan.parse(sheet_zon, header=2)
df_zon_raw['Codigo UDS'] = df_zon_raw['Codigo Unidad Servicio UDS'].astype(str).str.split('.').str[0].str.strip()
df_zon_agg = df_zon_raw.groupby(['Codigo UDS', 'SERVICIO 2026'])['Cupos'].sum().reset_index()


### 4. Integración y Fuzzy Matching de Servicios
Aplicamos la consolidación y el bucle de búsqueda difusa para empatar los 7 servicios de Regionales con los servicios del Metaverso bajo el mismo NIT.

In [4]:
print("Consolidando matriz granular por UDS con Fuzzy Matching...")

# Pre-agregamos Regionales con Financiero y Cuentame
res_base = df_reg.merge(df_comp_agg, on='id_clean', how='left')
res_base = res_base.merge(df_cue_agg, on='id_clean', how='left')
res_base = res_base.merge(df_cue_nit_map, on=['id_clean', 'reg_clean'], how='left')

# La base ahora será el Metaverso (UDS)
res = df_met[[
    'reg_clean', 'nit_clean', 'Codigo UDS', 'Nombre UDS', 
    'SERVICIO\n2026', 'Cupos a Programar 2026'
]].copy()

# Agregamos columnas de la base original para mantener compatibilidad
for col in res_base.columns:
    if col not in res.columns: res[col] = None

service_cols = ['Servicio 1', 'Servicio 2', 'Servicio 3', 'servicio 4', 'servicio 5', 'servicio 6', 'servicio 7']
attention_cols = ['Atenciones_1', 'Atenciones_2', 'Atenciones_3', 'Atenciones_4', 'Atenciones_5', 'Atenciones_6', 'Atenciones_7']

for idx, row in res.iterrows():
    nit = row['nit_clean']
    reg = row['reg_clean']
    uds_service = str(row['SERVICIO\n2026'])
    
    # Buscamos el contrato en Regionales que coincida con NIT y Regional
    match_reg = res_base[(res_base['nit_clean'] == nit) & (res_base['reg_clean'] == reg)]
    
    if not match_reg.empty:
        # Tomamos el primer contrato encontrado
        cont_row = match_reg.iloc[0]
        
        # Llenamos la info del contrato
        for col in res_base.columns:
            res.at[idx, col] = cont_row[col]
            
    # Cruce con Zonificación (por Codigo UDS)
    uds_code = row['Codigo UDS']
    zon_match = df_zon_agg[df_zon_agg['Codigo UDS'] == uds_code]
    if not zon_match.empty:
        res.at[idx, 'Atenciones Zonificacion'] = zon_match['Cupos'].sum()
    else:
        res.at[idx, 'Atenciones Zonificacion'] = 0

res['Cruza Compromisos'] = res['Valor Actual'].notna().apply(lambda x: 'SÍ' if x else 'NO')
res['Cruza Cuentame'] = res['Atenciones Cuentame'].notna().apply(lambda x: 'SÍ' if x else 'NO')
res['Total Atenciones Regionales'] = res[attention_cols].fillna(0).sum(axis=1)

# Reordenar para que Codigo y UDS estén al principio
cols = ['Codigo UDS', 'Nombre UDS', 'reg_clean', 'nit_clean'] + [c for c in res.columns if c not in ['Codigo UDS', 'Nombre UDS', 'reg_clean', 'nit_clean']]
res = res[cols]


Consolidando matriz con Fuzzy Matching para servicios via puente NIT...


Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons will have score 0. [Query: ' ']
Applied processor reduces input query to empty string, all comparisons wi

### 5. Generación de Reporte Formateado (Excel)
Aplicamos formato condicional (semaforización) para detectar inconsistencias directamente en el Excel.

In [5]:
output_path = "MATRIZ_MAESTRA_CONTROL_FINAL_24_ABRIL.xlsx"
writer = pd.ExcelWriter(output_path, engine='xlsxwriter')
res.to_excel(writer, sheet_name='Matriz Control', index=False)

workbook  = writer.book
worksheet = writer.sheets['Matriz Control']
format_red = workbook.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006'})
format_yellow = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C6500'})

# Alerta Compromisos (Rojo)
flag_idx = res.columns.get_loc('Cruza Compromisos')
worksheet.conditional_format(1, 0, len(res), len(res.columns)-1, {
    'type': 'formula',
    'criteria': f'=${xlsxwriter.utility.xl_col_to_name(flag_idx)}2="NO"',
    'format': format_red
})

# Alerta Discrepancia de Cupos (Amarillo)
for i in range(7):
    reg_idx = res.columns.get_loc(attention_cols[i])
    met_idx = res.columns.get_loc(f'Atenc_Metaverso_{i+1}')
    zon_idx = res.columns.get_loc(f'Atenc_Zonificacion_{i+1}')
    
    worksheet.conditional_format(1, reg_idx, len(res), reg_idx, {
        'type': 'formula',
        'criteria': f'=AND(ISNUMBER(${xlsxwriter.utility.xl_col_to_name(met_idx)}2), ${xlsxwriter.utility.xl_col_to_name(reg_idx)}2<>${xlsxwriter.utility.xl_col_to_name(met_idx)}2)',
        'format': format_yellow
    })
    worksheet.conditional_format(1, reg_idx, len(res), reg_idx, {
        'type': 'formula',
        'criteria': f'=AND(ISNUMBER(${xlsxwriter.utility.xl_col_to_name(zon_idx)}2), ${xlsxwriter.utility.xl_col_to_name(reg_idx)}2<>${xlsxwriter.utility.xl_col_to_name(zon_idx)}2)',
        'format': format_yellow
    })

writer.close()
print(f"¡Reporte generado exitosamente en {output_path}!")

¡Reporte generado exitosamente en MATRIZ_MAESTRA_CONTROL_FINAL_24_ABRIL.xlsx!
